# VQE-based Minimum Birkhoff Decomposition

This notebook walks through the **Birkhoff Decomposition** problem — decomposing a doubly stochastic matrix into a weighted sum of permutation matrices — using a custom VQE built on Divi.

**What you'll learn:**
1. How the Birkhoff decomposition problem is formulated
2. How to extend Divi's `VQE` class for a custom quantum-classical hybrid algorithm
3. How the classical `black_box_optimizer` works with quantum measurement outcomes
4. How to run on Qoro's cloud backend for larger problem instances

**Key Divi feature:** This example demonstrates the **modularity** of Divi's design — the entire VQE infrastructure (parameter management, gradient computation, circuit execution) is inherited, and only two methods need to be overridden.

## 1. Setup & Problem Data

We load a problem instance from the Quantum Optimization Benchmarking Library. Each instance provides:
- A **scaled doubly stochastic matrix** $D$ to decompose
- The **known solution** (permutations + weights) for validation

The goal: find permutation matrices $P_1, ..., P_k$ and weights $w_1, ..., w_k$ such that $D \approx \sum_i w_i P_i$.

In [ ]:
import json
import os

import numpy as np
import pennylane as qml

from birkhoff_vqe import BirkhoffDecomposition, combination_to_integer
from divi.backends import ParallelSimulator, QoroService, JobConfig
from divi.qprog.algorithms._ansatze import GenericLayerAnsatz
from divi.qprog.optimizers import ScipyOptimizer, ScipyMethod

### Choose Your Backend

Divi makes it easy to switch between **local simulation** and **Qoro's cloud backend**. Set `USE_CLOUD = True` to send your circuits to QoroService — no code changes needed anywhere else.

Get your API key at [dash.qoroquantum.net](https://dash.qoroquantum.net).

In [ ]:
USE_CLOUD = False  # Set to True to use QoroService cloud backend

if USE_CLOUD:
    # QoroService automatically reads QORO_API_KEY from your environment
    backend = QoroService(config=JobConfig(shots=10_000))
    print("☁️  Using QoroService cloud backend")
else:
    backend = ParallelSimulator(shots=10_000)
    print("💻 Using local ParallelSimulator")

In [ ]:
# Problem parameters
N_DIM = 3          # Matrix dimension (3×3 or 4×4)
K_COMBINATIONS = 2 # Number of permutations in the decomposition
INSTANCE_ID = "1"  # Problem instance (1-10)
MATRIX_TYPE = "sparse"  # "sparse" or "dense"
MAX_ITERATIONS = 10

### Load Problem Data

Each instance provides the target matrix and all possible permutation matrices for this dimension.

In [ ]:
dirname = os.path.dirname(os.path.abspath("__file__"))
mat_file = os.path.join(dirname, f"qbench_0{N_DIM}_{MATRIX_TYPE}.json")
perm_file = os.path.join(dirname, f"p{N_DIM}.dat")

with open(mat_file) as f:
    problem_data = json.load(f)
permutations = np.loadtxt(perm_file, dtype=int)

all_permutation_matrices = np.eye(N_DIM, dtype=int)[permutations - 1]

print(f"Loaded {len(all_permutation_matrices)} permutation matrices for dimension {N_DIM}")

In [ ]:
# Parse the specific instance
instance_data = problem_data[INSTANCE_ID]
n = instance_data["n"]
matrix = np.array(instance_data["scaled_doubly_stochastic_matrix"]).reshape((n, n))
sol_perms = np.eye(n, dtype=int)[
    np.array(instance_data["permutations"]).reshape(-1, n) - 1
]
scale = instance_data["scale"]
sol_weights = np.array(instance_data["weights"])

print(f"\nTarget matrix (scaled by {scale}):")
print(matrix)
print(f"\nKnown solution weights: {sol_weights / scale}")

## 2. Configure the VQE

We set up the custom `BirkhoffDecomposition` class, which **inherits from Divi's VQE** and only overrides two methods:

1. `_post_process_results` — connects quantum measurement outcomes to the classical optimizer
2. The classical `black_box_optimizer` — contains the Birkhoff-specific optimization logic

All other VQE machinery (circuits, parameter tuning, backend communication) is inherited automatically. The `backend` parameter works identically whether it's `ParallelSimulator` or `QoroService`.

In [ ]:
# Optimizer for VQE parameter tuning
optimizer = ScipyOptimizer(ScipyMethod.COBYLA)

# Ansatz (parameterized quantum circuit)
ansatz = GenericLayerAnsatz([qml.RY], entangler=qml.CZ, entangling_layout="brick")

In [ ]:
birkhoff_vqe = BirkhoffDecomposition(
    matrix=matrix,
    scale=scale,
    n=N_DIM,
    k=K_COMBINATIONS,
    all_perms_matrix=all_permutation_matrices,
    backend=backend,  # Uses QoroService or ParallelSimulator from above
    optimizer=optimizer,
    max_iterations=MAX_ITERATIONS,
    ansatz=ansatz,
    n_layers=3,
)

print(f"VQE configured: {birkhoff_vqe.n_qubits} qubits, {MAX_ITERATIONS} iterations")

## 3. Run the VQE

The VQE loop:
1. Proposes ansatz parameters
2. Runs the quantum circuit → measurement outcomes
3. Post-processes: maps bitstrings to permutation combinations, runs the classical optimizer
4. Updates the cost and repeats

In [ ]:
print("Starting VQE optimization...")
birkhoff_vqe.run()

## 4. Analyze Results

After VQE convergence, we extract the best decomposition and compare it to the known solution.

In [ ]:
combo, weights = birkhoff_vqe.find_final_decomposition()

if combo and weights:
    print("VQE Decomposition:")
    for perm_idx, weight in zip(combo, weights):
        if weight > 1e-6:
            print(f"  Permutation {perm_idx}: weight = {weight:.6f}")

    # Reconstruct matrix from VQE solution
    vqe_reconstructed = sum(
        weight * all_permutation_matrices[perm_idx]
        for perm_idx, weight in zip(combo, weights)
        if weight > 1e-6
    )
    original_unscaled = matrix / scale

    error = np.linalg.norm(original_unscaled - vqe_reconstructed, ord=2)
    print(f"\nOriginal matrix (unscaled):\n{original_unscaled}")
    print(f"\nVQE reconstructed matrix:\n{vqe_reconstructed}")
    print(f"\nL2 error: {error:.6f}")
else:
    print("VQE did not find a valid decomposition.")

### Solution Probability

We can also check how often the VQE's final quantum state produced the correct bitstring — higher probability means the VQE is more confident in this solution.

In [ ]:
if combo and weights:
    total_shots = sum(birkhoff_vqe.final_measurement_outcomes.values())
    solution_integer = combination_to_integer(combo, birkhoff_vqe.k)
    solution_bitstring = format(solution_integer, f"0{birkhoff_vqe.n_qubits}b")
    solution_shots = birkhoff_vqe.final_measurement_outcomes.get(solution_bitstring, 0)

    if total_shots > 0:
        probability = (solution_shots / total_shots) * 100
        print(f"Solution bitstring: '{solution_bitstring}'")
        print(f"Measured {solution_shots}/{total_shots} times ({probability:.2f}%)")

## Understanding the Design

The power of this example lies in how little custom code was needed:

| Component | Source | Lines of Custom Code |
|-----------|--------|---------------------|
| VQE optimization loop | Inherited from Divi | 0 |
| Circuit construction | Inherited from Divi | 0 |
| Backend communication | Inherited from Divi | 0 |
| Post-processing hook | Custom override | ~50 lines |
| Classical optimizer | Custom implementation | ~80 lines |

Note: switching from `ParallelSimulator` to `QoroService` requires **zero code changes** — just flip `USE_CLOUD = True` above.

See `birkhoff_vqe.py` for the full implementation of `BirkhoffDecomposition`.

## Citation

> G. S. Barron, et al., "Quantum Optimization Benchmarking Library: The Intractable Decathlon," arXiv:2504.03832 [quant-ph], (2025). https://arxiv.org/abs/2504.03832